# Quantizing a 0.5B Real LLM from Scratch (Qwen 2.5 0.5B)

> **Week 11 Lecture Demo** — CS 203: Software Tools and Techniques for AI · IIT Gandhinagar

We have now quantized:

- Notebook 09 — a **tiny MLP** on `make_moons`
- Notebook 11 — a **60K-param transformer** we trained from scratch
- **This notebook** — a **494M-param open-source LLM** (Qwen 2.5 0.5B) we download from the Hugging Face Hub

The quantization routine is **the same 3-line function** we wrote on day one. Zero changes. That is the whole point.

> **Runtime recommendation:** turn on the T4 GPU in Colab (*Runtime → Change runtime type → T4*). CPU works but generation of 50 tokens can take ~30s per model.


In [ ]:
!pip install -q -U transformers accelerate


In [ ]:
import math, copy, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.float32       # keep everything FP32 so the quantization error is purely from INT8
print(f"device: {device}")


---

## 1. Load the model and tokenizer

`Qwen/Qwen2.5-0.5B` — 494M parameters, Apache-2.0 licensed, no gated access required. Loading in FP32 takes ~2 GB of RAM.


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype).to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"loaded {MODEL_ID}")
print(f"total parameters: {n_params:,}  (~{n_params/1e6:.1f}M)")


### Where are the `nn.Linear` layers?

In every decoder block: `self_attn.{q,k,v,o}_proj` for attention + `mlp.{gate,up,down}_proj` for the SwiGLU MLP. Plus `lm_head` at the top. Let's enumerate them.


In [ ]:
linear_layers = [(n, m) for n, m in model.named_modules() if isinstance(m, nn.Linear)]
print(f"number of nn.Linear layers: {len(linear_layers)}")

# Count parameters in linear layers vs total
lin_params = sum(m.weight.numel() + (m.bias.numel() if m.bias is not None else 0)
                 for _, m in linear_layers)
print(f"parameters in Linear layers: {lin_params:,}  ({lin_params/n_params*100:.1f}% of model)")
print()
print("first few linear layers:")
for n, m in linear_layers[:6]:
    print(f"  {n:50s}  in={m.in_features}  out={m.out_features}")


Almost every parameter in the model lives inside these `nn.Linear` layers — so quantizing them to INT8 shrinks the model by ~4× even though we leave embeddings and LayerNorm alone.


---

## 2. Baseline FP32 generation

Greedy decoding for determinism. Short prompt, 50 new tokens.


In [ ]:
PROMPT = "The key idea of quantization is"

def generate(m, prompt, max_new_tokens=50, greedy=True):
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        out = m.generate(
            ids,
            max_new_tokens=max_new_tokens,
            do_sample=not greedy,
            temperature=1.0 if greedy else 0.8,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)


t0 = time.time()
fp32_text = generate(model, PROMPT, max_new_tokens=50, greedy=True)
t_fp32 = time.time() - t0
print(f"[FP32, {t_fp32:.2f}s]")
print(fp32_text)


---

## 3. INT8 symmetric quantization — the same three-line routine

Literally copy-paste from notebooks 09 and 11. The model doesn't care that it's 494M params instead of 60K.


In [ ]:
def quantize_tensor(x):
    """Symmetric per-tensor INT8 quantization."""
    max_abs = x.abs().max().item()
    if max_abs == 0:
        return torch.zeros_like(x, dtype=torch.int8), 1.0
    scale = max_abs / 127.0
    q = torch.round(x / scale).clamp(-127, 127).to(torch.int8)
    return q, scale


def dequantize_tensor(q, scale):
    return q.to(torch.float32) * scale


### Apply it to every `nn.Linear` in the LLM

To keep comparisons fair we **don't touch** `model` directly — we deep-copy it and modify the copy. (Warning: a deep copy of a 0.5B FP32 model uses another ~2 GB. On a low-RAM runtime, you can instead iterate in-place.)


In [ ]:
def quantize_all_linears(net, skip_lm_head=False):
    """Replace each nn.Linear weight with its fake-quantized FP32 equivalent.
    Returns a fresh deep-copied network + a report: (name, shape, scale, max_err)."""
    qnet = copy.deepcopy(net)
    report = []
    for name, mod in qnet.named_modules():
        if not isinstance(mod, nn.Linear):
            continue
        if skip_lm_head and "lm_head" in name:
            continue
        w_orig = net.get_submodule(name).weight.data
        q, s = quantize_tensor(mod.weight.data)
        mod.weight.data = dequantize_tensor(q, s)
        mod._int8_weight = q        # keep the int8 tensor (1 byte/param) for size accounting
        mod._scale = s
        err = (w_orig - mod.weight.data).abs().max().item()
        report.append((name, tuple(mod.weight.shape), s, err))
    return qnet, report


print("quantizing... (this takes ~10-20s on CPU, a few seconds on GPU)")
qmodel, report = quantize_all_linears(model)
qmodel.eval()
print(f"quantized {len(report)} Linear layers")


---

## 4. INT8 generation — identical prompt, identical decoding

If quantization is working, the INT8 model should generate *very similar* text — often character-for-character identical for the first few tens of tokens under greedy decoding.


In [ ]:
t0 = time.time()
int8_text = generate(qmodel, PROMPT, max_new_tokens=50, greedy=True)
t_int8 = time.time() - t0
print(f"[INT8, {t_int8:.2f}s]")
print(int8_text)


In [ ]:
print("FP32\n" + "-" * 70)
print(fp32_text)
print("\nINT8\n" + "-" * 70)
print(int8_text)
print("\n(If they look nearly identical for the first several tokens, the quantizer is doing its job.)")


---

## 5. Model size

Linear weights dominate a 0.5B LLM. Quantizing them to INT8 drops those to ~¼ their original size; embeddings and LayerNorm stay FP32.


In [ ]:
def count_bytes(net, quantized_linears=False):
    total = 0
    seen = set()
    # Count all parameters; Linear weights use 1 byte when quantized + 4 bytes for one scale.
    for name, mod in net.named_modules():
        for pname, p in mod.named_parameters(recurse=False):
            full = f"{name}.{pname}" if name else pname
            if full in seen:
                continue
            seen.add(full)
            if quantized_linears and isinstance(mod, nn.Linear) and pname == "weight":
                total += p.numel() * 1   # int8
                total += 4               # one FP32 scale
            else:
                total += p.numel() * 4   # FP32
    return total


size_fp32 = count_bytes(model, quantized_linears=False)
size_int8 = count_bytes(qmodel, quantized_linears=True)
print(f"FP32 model size:  {size_fp32/1e6:8.1f} MB")
print(f"INT8 model size:  {size_int8/1e6:8.1f} MB   ({size_fp32/size_int8:.2f}x smaller)")


A 0.5B FP32 model is ~2 GB. Quantized to INT8 (on linear weights only) it drops to ~0.5–0.6 GB — fits comfortably in GPU memory on consumer cards, and more importantly, fits in the memory of a phone / edge device.


---

## 6. Next-token log-likelihood — a harder test than "does the text look right"

Text samples can hide quantization damage. A better check: feed the FP32 and INT8 models the **same** long paragraph and compute next-token cross-entropy. If this jumps meaningfully, something is wrong.


In [ ]:
EVAL_TEXT = (
    "Quantization is a central technique in modern machine learning systems. "
    "It reduces the memory footprint of a neural network by representing its "
    "weights and activations with fewer bits than the native floating-point "
    "format. A well-quantized model can run on phones, browsers, and edge "
    "devices that would otherwise be too resource-constrained for full "
    "precision inference. "
) * 4

ids = tokenizer(EVAL_TEXT, return_tensors="pt").input_ids.to(device)

@torch.no_grad()
def next_token_ce(m, ids):
    logits = m(ids).logits
    return F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)),
                           ids[:, 1:].reshape(-1)).item()

ce_fp32 = next_token_ce(model, ids)
ce_int8 = next_token_ce(qmodel, ids)
print(f"FP32 next-token CE: {ce_fp32:.4f}")
print(f"INT8 next-token CE: {ce_int8:.4f}")
print(f"relative change:    {(ce_int8 - ce_fp32) / ce_fp32 * 100:+.3f}%")


For a symmetric per-tensor INT8 scheme on a 0.5B model, the CE typically moves by **well under 1%** — unnoticeable for most use cases. This is why INT8 is the default production precision for LLMs.


---

## 7. Where is most of the error?

Plot per-layer round-trip error. In real LLMs, certain layers — especially down-projections at the end of MLPs, and some attention `o_proj` — have larger weight ranges and thus larger per-weight rounding errors.


In [ ]:
import matplotlib.pyplot as plt

names  = [r[0].replace("model.layers.", "L").replace(".self_attn.", ".a.").replace(".mlp.", ".m.") for r in report]
errors = [r[3] for r in report]

plt.figure(figsize=(12, 4))
plt.plot(errors, marker=".", linewidth=0.8)
plt.xlabel("linear layer index (in traversal order)")
plt.ylabel("max |W - dequant(q)|")
plt.title(f"Per-layer round-trip INT8 quantization error — {MODEL_ID}")
plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Which layers have the largest rounding error?
top = sorted(enumerate(errors), key=lambda x: -x[1])[:5]
print("top 5 noisiest layers:")
for idx, err in top:
    print(f"  {names[idx]:60s}  max_err={err:.6f}  scale={report[idx][2]:.6f}")


---

## 8. Exercises

1. **Skip `lm_head`.** The output projection is often the most sensitive Linear in an LLM (it writes directly to the vocab logits). Re-run `quantize_all_linears(model, skip_lm_head=True)` and recompute the CE. Does it get measurably closer to FP32?

2. **Per-channel quantization.** Change `quantize_tensor` to take `dim=0` and compute one scale per *output row*. Apply to this LLM. How much does the CE improve? (Industry INT8 schemes — TensorRT, bitsandbytes — default to per-channel for exactly this reason.)

3. **Quantize `nn.Embedding` too.** The token embedding table in Qwen 2.5 0.5B is `vocab_size × hidden` — it's a single massive FP32 tensor. Quantize it with the same routine. Does the CE move? (It probably does; the embedding table is often the biggest single tensor in a small LLM and has very wide ranges due to rare tokens.)

4. **INT4 instead of INT8.** Change `clamp(-127, 127)` to `clamp(-7, 7)`. Re-run generation. At what point does the text start to visibly degrade? Does one-shot INT4 without calibration collapse generation, and if so, does per-channel quant rescue it?

5. **A smaller model.** Swap `MODEL_ID` to `HuggingFaceTB/SmolLM2-135M` (or any model ≤ 500M params). Does the CE-delta get larger or smaller on a smaller model? Why might smaller models be *more* sensitive to quantization?

6. **Time it properly.** Our INT8 model is still running FP32 matmuls under the hood (we dequantized before the forward pass). To get real speedups you'd need INT8 matmul kernels — look up `torch.int_mm` or `bitsandbytes.nn.Linear8bitLt`. How much faster is actual INT8 matmul on your hardware?

---

## Recap

The same `quantize_tensor(x) = round(x / (max|x|/127))` that worked on a 60K-param toy also works on a 494M-param open-source LLM, with **<1% next-token loss increase** and **~4× memory reduction on the weight matrices that dominate the model**. That is all symmetric per-tensor INT8 quantization is. Production systems (GPTQ, AWQ, SmoothQuant, llama.cpp's GGUF) layer calibration, outlier handling, per-group scales, and INT8 matmul kernels on top of this same three-line core.
